In [1]:
# 1. Installations and Dependencies

# Core Agents & AI
# %pip install -qU langchain-openai
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv
# %pip install -qU tavily-python

# Web Scraping Tools
# %pip install -qU ddgs
# %pip install -qU selenium
# %pip install -qU webdriver-manager
# %pip install -qU beautifulsoup4

#Memory and data persistence

# %pip install -qU aiosqlite
# %pip install -qU langgraph-checkpoint-sqlite

print("[✅] Dependencies configuration checked.")

[✅] Dependencies configuration checked.


In [2]:
# 2. Environment setup: Local LLM (Llama 3 Power)
from langchain_openai import ChatOpenAI

# LM Studio Configuration
lm_studio_base = "http://localhost:1234/v1"
lm_studio_key = "lm-studio" 

# Initialize the LLM
# DICA: Certifique-se que o "Context Length" no LM Studio está setado para 8192 ou mais!
llm = ChatOpenAI(
    model="meta-llama-3.1-8b-instruct", # Atualizado para o Llama 3
    base_url=lm_studio_base,
    api_key=lm_studio_key,
    temperature=0
)

print(f"Target Model: meta-llama-3.1-8b-instruct at {lm_studio_base}")
try:
    response = llm.invoke("System check. Reply 'Online'.").content
    print(f"[✅] Local LLM Status: {response}")
except Exception as e:
    print(f"[❌] Local LLM Connection failed: {e}")

Target Model: meta-llama-3.1-8b-instruct at http://localhost:1234/v1
[✅] Local LLM Status: Online.


In [3]:
# 3. Environment setup: Tavily Search Client
import os
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

# Verify API Key
tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env file.")

# Initialize Client
tavily_client = TavilyClient(tavily_api_key)

print("[✅] Tavily Client initialized.")

# Optional: Quick Connectivity Test
# try:
#     test_response = tavily_client.search(query="test connectivity", max_results=1)
#     if test_response and 'results' in test_response:
#         print("[✅] Tavily API connection successful.")
#     else:
#         print("[⚠️] Tavily connected but returned no results.")
# except Exception as e:
#     print(f"[❌] Tavily Connection failed: {e}")

[✅] Tavily Client initialized.


In [4]:
# 4. Persistence Setup (SQLite Checkpointer)
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Connect to a local SQLite database to store conversation history
# check_same_thread=False is required for SQLite in this context
db_connection = sqlite3.connect("langgraph_memory.db", check_same_thread=False)

# The 'memory' object is the checkpointer that the graph will use
# It saves the state of the conversation after each step
memory = SqliteSaver(conn=db_connection)

print("[✅] SQLite database connection and checkpointer ready.")

[✅] SQLite database connection and checkpointer ready.


In [5]:
#5. Custom function that searches for messages with the same ID

from typing import TypedDict, Annotated, List, Dict, Any, Optional
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import (
    AnyMessage, SystemMessage, HumanMessage, ToolMessage, BaseMessage
)

from uuid import uuid4

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:

    for message in right:
        if not message.id:
            message.id = str(uuid4())

    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged


# Updated version of AgentState for the new function

# class AgentState(TypedDict):
#     messages: Annotated[list[AnyMessage], reduce_messages]


print("[✅] reduceMessages is ready") 

[✅] reduceMessages is ready


In [6]:
# 6. Thread Initialization & Logging Utilities
# --------------------------------------------
# This cell ONLY prepares:
# - a dynamic thread id
# - logging helpers
# - safe snapshot inspection utilities
# It does NOT assume that the graph has already run.

import uuid
from typing import Any, Dict

# ----------------------------
# Thread identity
# ----------------------------
dynamic_thread_id = str(uuid.uuid4())
print(f"[🧵] Dynamic Thread ID created: {dynamic_thread_id}")

# Default thread config (used later by the graph)
thread_config = {
    "configurable": {
        "thread_id": dynamic_thread_id
    }
}

# ----------------------------
# Logging helpers
# ----------------------------
def log_section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def log_kv(key: str, value: Any):
    print(f"[LOG] {key}: {value}")

# ----------------------------
# Snapshot metadata extractor (SAFE)
# ----------------------------
def extract_snapshot_metadata(snapshot) -> Dict[str, Any]:
    """
    Safely extract metadata from a LangGraph state snapshot.
    This function NEVER assumes the snapshot exists.
    """
    meta = {
        "thread_id": None,
        "thread_ts": None,
        "run_id": None,
        "has_snapshot": False,
    }

    if snapshot is None:
        return meta

    meta["has_snapshot"] = True

    try:
        cfg = getattr(snapshot, "config", {}) or {}
        configurable = cfg.get("configurable", {})

        meta["thread_id"] = configurable.get("thread_id")
        meta["run_id"] = configurable.get("__run_id")
        meta["thread_ts"] = configurable.get("thread_ts")

    except Exception as e:
        meta["error"] = str(e)

    return meta

# ----------------------------
# Placeholder (IMPORTANT)
# ----------------------------
# This variable will ONLY be populated AFTER the graph pauses.
# Do NOT try to read it yet.
current_state_snapshot = None

print("[✅] Logging utilities and thread context initialized.")

[🧵] Dynamic Thread ID created: d5d4103e-c5c8-4a05-acf6-7acfa6444c04
[✅] Logging utilities and thread context initialized.


In [7]:
# Cell 7: Updated with the new v1.0 agent pattern
from typing import TypedDict, Annotated, List
from langchain.agents import create_agent  # Updated import [citation:2][citation:6]
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

# Define the agent state (unchanged)
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], reduce_messages]

print("[✅] HumanInTheLoopMiddleware imported and ready to use.")

[✅] HumanInTheLoopMiddleware imported and ready to use.


In [21]:
# 8. Main Execution Flow with Human-in-the-Loop (Two-Stage Validation)
# ---------------------------------------------------------------------
# This cell implements complete workflow with TWO human validation points:
# STAGE 1: Validate search query BEFORE execution (Approve/Edit/Reject)
# STAGE 2: Validate final answer AFTER generation (Approve/Correction)
# If answer is rejected, human provides correction and LLM reprocesses

from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from datetime import datetime
import uuid
from langgraph.types import Command
import sqlite3

# ----------------------------
# Tool definition (Tavily)
# ----------------------------
@tool
def tavily_search(query: str) -> str:
    """Search the web for current information. Returns search results as a string."""
    search_tool = TavilySearch(max_results=3)
    try:
        result = search_tool.invoke(query)
        return str(result)
    except Exception as e:
        return f"Search error: {str(e)}"

# ----------------------------
# System prompt
# ----------------------------
current_date = datetime.now().strftime("%Y-%m-%d")
tool_name = tavily_search

SYSTEM_PROMPT = f"""
You are an intelligent research assistant. Today's date is {current_date}.


Workflow:
1. Use the search engine ({tool_name}) to gather up-to-date information.
2. Before executing any search, pause and present a concise proposed query for human approval.
3. After the search runs, present tool results and a concise draft answer for human validation.
4. If the human provides a correction:
- Treat it as a proposed correction, not ground truth.
- Check for conflicts with high-confidence or well-established information.
- If a conflict exists, explicitly state it.
- Ask for explicit confirmation using: OVERRIDE = YES
- Only apply the correction fully if OVERRIDE = YES is given.
- Otherwise, preserve factual accuracy.


Output rules:
- Always include a short "SOURCES" list when external information is used.
- Always include a one-line "CONFIDENCE" estimate (low/medium/high).
- If a human override is applied, clearly label the answer as HUMAN_OVERRIDE.
- Never fabricate sources to justify a human correction.


Keep answers concise, critical, and auditable.
""".strip()

# ----------------------------
# Create the agent with HITL middleware
# ----------------------------
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model=llm,
    tools=[tavily_search],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "tavily_search": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                }
            },
            description_prefix="Search execution pending approval",
        )
    ]
)

print("[✅] Agent created with Two-Stage Human Validation workflow.")

# ----------------------------
# Thread setup
# ----------------------------
THREAD_ID = f"2stage-{uuid.uuid4().hex[:8]}"
thread_config = {"configurable": {"thread_id": THREAD_ID}}

print("\n" + "=" * 80)
print(f"🚀 THREAD ID: {THREAD_ID}")
print("=" * 80)

# ----------------------------
# User query
# ----------------------------
query = "What is the distance between Tokyo and Rio de Janeiro?"
print(f"\n💬 USER QUERY: {query}")

# ----------------------------
# STAGE 1 — Run agent (pauses at tool call for query validation)
# ----------------------------
print("\n" + "=" * 80)
print("🔍 STAGE 1: Human validates SEARCH QUERY before execution")
print("=" * 80)

# Initial input
initial_input = {"messages": [HumanMessage(content=query)]}

# Collect events and look for interrupt
interrupt_detected = False
interrupt_data = None
search_query = ""

print("[🔍] Streaming agent execution...")
for event in agent.stream(initial_input, config=thread_config, stream_mode="updates"):
    for node_name, node_data in event.items():
        if node_name == "__interrupt__":
            interrupt_detected = True
            interrupt_data = node_data
            print("\n⏸️ INTERRUPT DETECTED: Tool call requires human approval")

if not interrupt_detected:
    print("\n⚠️ No interrupt detected - the agent completed without tool calls.")
    final_state = agent.get_state(thread_config)
    if final_state and "messages" in final_state.values:
        ai_messages = [msg for msg in final_state.values["messages"] if isinstance(msg, AIMessage)]
        if ai_messages:
            print("\n[🤖] Agent response without search:")
            print(ai_messages[-1].content)
            # Skip to Stage 2 validation
            final_answer = ai_messages[-1].content
            interrupt_detected = False  # We'll handle as direct answer

# ----------------------------
# Stage 1 Human Decision (if interrupt occurred)
# ----------------------------
if interrupt_detected and interrupt_data:
    # Extract interrupt data
    if isinstance(interrupt_data, tuple) and len(interrupt_data) >= 1:
        interrupt_content = interrupt_data[0]
        
        if isinstance(interrupt_content, dict):
            action_requests = interrupt_content.get('action_requests', [])
        else:
            action_requests = getattr(interrupt_content, 'action_requests', [])
        
        if action_requests:
            action = action_requests[0]
            if isinstance(action, dict):
                args = action.get('arguments', {})
            else:
                args = getattr(action, 'arguments', {})
            
            search_query = args.get('query', "")
    
    print("\n" + "=" * 60)
    print("👤 STAGE 1: HUMAN VALIDATION OF SEARCH QUERY")
    print("=" * 60)
    
    if search_query:
        print(f"\n📋 Proposed search query:")
        print(f"   '{search_query}'")
    else:
        print("\n📋 No specific search query extracted.")
    
    print("\nOptions:")
    print("1. ✅ APPROVE - Execute the search as planned")
    print("2. ✏️ EDIT    - Modify the search query")
    print("3. ❌ REJECT  - Cancel the search and provide feedback")
    
    decision = input("\nChoose an option (1=approve, 2=edit, 3=reject): ").strip()
    
    if decision == "1":
        print("[✅] Human approved the search. Executing...")
        decisions = [{"type": "approve"}]
        
    elif decision == "2":
        print("\n✏️ EDIT MODE")
        if not search_query:
            search_query = input("Enter search query: ").strip()
        else:
            print(f"Current query: '{search_query}'")
            new_query = input("Enter new search query (or press Enter to keep current): ").strip()
            if new_query:
                search_query = new_query
                print(f"[✅] Query edited to: '{search_query}'")
        
        decisions = [{
            "type": "edit",
            "edited_action": {
                "name": "tavily_search",
                "args": {"query": search_query}
            }
        }]
            
    elif decision == "3":
        print("\n❌ REJECT MODE")
        feedback = input("Provide feedback/instructions for the agent: ").strip()
        
        if feedback:
            print(f"[✅] Feedback provided: '{feedback}'")
            decisions = [{
                "type": "reject",
                "message": feedback
            }]
        else:
            decisions = [{
                "type": "reject",
                "message": "The search was rejected. Please respond to the user without performing a search."
            }]
    else:
        print("[⚠️] Invalid choice. Defaulting to approval.")
        decisions = [{"type": "approve"}]
    
    # ----------------------------
    # Resume after Stage 1 decision
    # ----------------------------
    print("\n" + "=" * 80)
    print("🔄 RESUMING AFTER STAGE 1 DECISION")
    print("=" * 80)
    
    # Resume execution with the human decision
    answer_received = False
    final_answer = ""
    
    try:
        for event in agent.stream(
            Command(resume={"decisions": decisions}),
            config=thread_config,
            stream_mode="updates"
        ):
            for node_name, node_data in event.items():
                if node_name == "agent" and isinstance(node_data, dict) and "messages" in node_data:
                    response_messages = node_data["messages"]
                    ai_responses = [msg for msg in response_messages if isinstance(msg, AIMessage) and hasattr(msg, 'content') and msg.content]
                    if ai_responses:
                        final_answer = ai_responses[-1].content
                        answer_received = True
                        break
            if answer_received:
                break
    except Exception as e:
        print(f"[❌] Error resuming execution: {e}")
    
    # Fallback: check final state
    if not answer_received:
        final_state = agent.get_state(thread_config)
        if final_state and "messages" in final_state.values:
            ai_messages = [msg for msg in final_state.values["messages"] if isinstance(msg, AIMessage)]
            if ai_messages and ai_messages[-1].content:
                final_answer = ai_messages[-1].content

# ----------------------------
# STAGE 2 — Human validates final answer
# ----------------------------
print("\n" + "=" * 80)
print("🔍 STAGE 2: Human validates FINAL ANSWER for coherence")
print("=" * 80)

if final_answer:
    print("\n🤖 AGENT'S ANSWER:")
    print("-" * 60)
    print(final_answer)
    print("-" * 60)
    
    print("\n" + "=" * 60)
    print("👤 STAGE 2: HUMAN VALIDATION OF FINAL ANSWER")
    print("=" * 60)
    
    print("\nIs this answer coherent and correct?")
    print("1. ✅ YES - Answer is correct (process ends)")
    print("2. ❌ NO  - Answer needs correction (provide correct info)")
    
    validation_decision = input("\nChoose (1=yes, 2=no): ").strip()
    
    if validation_decision == "2":
        # Human provides correction
        print("\n✏️ CORRECTION MODE")
        correction = input("Please provide the correct information: ").strip()
        
        if correction:
            print(f"[✅] Correction received: '{correction}'")
            
            # Prepare new input with correction context
            correction_context = f"The previous answer was not correct. Here is the correct information: {correction}. Please provide a revised answer based on this corrected information."
            
            # Get current state to maintain conversation history
            current_state = agent.get_state(thread_config)
            if current_state and "messages" in current_state.values:
                # Add correction as new human message
                current_messages = current_state.values["messages"]
                current_messages.append(HumanMessage(content=correction_context))
                
                # Create a simplified prompt for reprocessing (no tools)
                reprocess_prompt = f"""Based on the human correction below, provide a revised answer.
                
                Human Correction: {correction}
                
                Original Question: {query}
                
                Please provide an accurate, concise answer using only the human-corrected information."""
                
                # Send to LLM for reprocessing
                print("\n" + "=" * 80)
                print("🔄 REPROCESSING WITH HUMAN CORRECTION")
                print("=" * 80)
                
                try:
                    # Call LLM directly with correction
                    messages_for_llm = [
                        SystemMessage(content="You are an assistant that revises answers based on human corrections."),
                        HumanMessage(content=f"Original question: {query}"),
                        AIMessage(content=f"Original answer (incorrect): {final_answer}"),
                        HumanMessage(content=f"Human correction: {correction}. Please provide a revised answer.")
                    ]
                    
                    revised_response = llm.invoke(messages_for_llm)
                    
                    if hasattr(revised_response, 'content'):
                        revised_answer = revised_response.content
                        print("\n🤖 REVISED ANSWER (based on human correction):")
                        print("=" * 60)
                        print(revised_answer)
                        print("=" * 60)
                        
                        # Update final answer
                        final_answer = revised_answer
                    else:
                        print("[❌] Could not get revised answer from LLM")
                        
                except Exception as e:
                    print(f"[❌] Error during reprocessing: {e}")
            else:
                print("[❌] Could not retrieve current state for reprocessing")
        else:
            print("[⚠️] No correction provided. Keeping original answer.")
    
    else:
        # Answer approved
        print("\n[✅] Human approved the answer. Process completed.")
        
else:
    print("\n[⚠️] No final answer generated to validate.")

# ----------------------------
# Final Summary
# ----------------------------
print("\n" + "=" * 80)
print("📊 FINAL EXECUTION SUMMARY")
print("=" * 80)

print("\nTwo-Stage Validation Workflow:")
print("1. ✅ STAGE 1: Human validated search query before execution")
if interrupt_detected:
    print("   - Query was: " + ("approved" if decision == "1" else "edited" if decision == "2" else "rejected"))
print("2. ✅ STAGE 2: Human validated final answer for coherence")
if 'validation_decision' in locals():
    print(f"   - Answer was: {'approved' if validation_decision == '1' else 'corrected and reprocessed'}")

print(f"\n📝 Final Answer:")
print("-" * 60)
print(final_answer if final_answer else "[No answer generated]")
print("-" * 60)

# ----------------------------
# Memory verification (fixed SQL query)
# ----------------------------
print("\n" + "=" * 80)
print("🧠 MEMORY VERIFICATION")
print("=" * 80)

try:
    # Get state history
    history = list(agent.get_state_history(thread_config))
    print(f"[✅] Thread '{THREAD_ID}' contains {len(history)} persisted steps.")
    
    # Fixed SQL query without thread_ts
    conn = sqlite3.connect("langgraph_memory.db")
    cursor = conn.cursor()
    
    # First, check what columns exist
    cursor.execute("PRAGMA table_info(checkpoints)")
    columns = cursor.fetchall()
    column_names = [col[1] for col in columns]
    
    if 'thread_ts' in column_names:
        cursor.execute("""
            SELECT thread_id, thread_ts FROM checkpoints 
            WHERE config LIKE ?
            ORDER BY thread_ts DESC

            
        """, (f'%{THREAD_ID}%',))
    else:
        # Use id or other available column
        cursor.execute("""
            SELECT thread_id, id FROM checkpoints 
            WHERE config LIKE ?
            ORDER BY id DESC
        """, (f'%{THREAD_ID}%',))
    
    checkpoints = cursor.fetchall()
    print(f"[✅] Database contains {len(checkpoints)} checkpoints for this thread.")
    
    conn.close()
except Exception as e:
    print(f"[⚠️] Memory verification issue: {e}")

print("\n[🎉] Two-Stage Human Validation Workflow Completed!")

[✅] Agent created with Two-Stage Human Validation workflow.

🚀 THREAD ID: 2stage-d9ebebf7

💬 USER QUERY: What is the distance between Tokyo and Rio de Janeiro?

🔍 STAGE 1: Human validates SEARCH QUERY before execution
[🔍] Streaming agent execution...

⏸️ INTERRUPT DETECTED: Tool call requires human approval

👤 STAGE 1: HUMAN VALIDATION OF SEARCH QUERY

📋 No specific search query extracted.

Options:
1. ✅ APPROVE - Execute the search as planned
2. ✏️ EDIT    - Modify the search query
3. ❌ REJECT  - Cancel the search and provide feedback



Choose an option (1=approve, 2=edit, 3=reject):  1


[✅] Human approved the search. Executing...

🔄 RESUMING AFTER STAGE 1 DECISION

🔍 STAGE 2: Human validates FINAL ANSWER for coherence

🤖 AGENT'S ANSWER:
------------------------------------------------------------
The distance between Tokyo and Rio de Janeiro is approximately 11,532 miles.

SOURCES:
* Prokerala
* Trip.com
* Travelmath

CONFIDENCE: medium

HUMAN_OVERRIDE: None
------------------------------------------------------------

👤 STAGE 2: HUMAN VALIDATION OF FINAL ANSWER

Is this answer coherent and correct?
1. ✅ YES - Answer is correct (process ends)
2. ❌ NO  - Answer needs correction (provide correct info)



Choose (1=yes, 2=no):  2



✏️ CORRECTION MODE


Please provide the correct information:  IT'S 530 kilometers


[✅] Correction received: 'IT'S 530 kilometers'

🔄 REPROCESSING WITH HUMAN CORRECTION

🤖 REVISED ANSWER (based on human correction):
Revised answer: The distance between Tokyo and Rio de Janeiro is approximately 530 kilometers.

Note: This seems to be an incorrect statement as the actual distance between Tokyo, Japan and Rio de Janeiro, Brazil is around 17,000-18,000 km (10,563 miles) depending on the route taken.

📊 FINAL EXECUTION SUMMARY

Two-Stage Validation Workflow:
1. ✅ STAGE 1: Human validated search query before execution
   - Query was: approved
2. ✅ STAGE 2: Human validated final answer for coherence
   - Answer was: corrected and reprocessed

📝 Final Answer:
------------------------------------------------------------
Revised answer: The distance between Tokyo and Rio de Janeiro is approximately 530 kilometers.

Note: This seems to be an incorrect statement as the actual distance between Tokyo, Japan and Rio de Janeiro, Brazil is around 17,000-18,000 km (10,563 miles) depend

In [15]:
# Simple visualization
mermaid_code = agent.get_graph().draw_mermaid()
print(mermaid_code)

# Or with IPython display
from IPython.display import display, Image
import base64

mermaid_encoded = base64.b64encode(mermaid_code.encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{mermaid_encoded}"))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	HumanInTheLoopMiddleware\2eafter_model(HumanInTheLoopMiddleware.after_model)
	__end__([<p>__end__</p>]):::last
	HumanInTheLoopMiddleware\2eafter_model -.-> __end__;
	HumanInTheLoopMiddleware\2eafter_model -.-> model;
	HumanInTheLoopMiddleware\2eafter_model -.-> tools;
	__start__ --> model;
	model --> HumanInTheLoopMiddleware\2eafter_model;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

